# Cohort and baseline EDA

Reproduces cohort, label, quality, baseline, and domain-shift analyses from local derived data. Saved aggregate tables can be reviewed without running this notebook.

Run the setup cell first. Full processing and training require locally provisioned datasets; the grading quick start uses synthetic tests and saved aggregate results.


In [ ]:
from pathlib import Path
import sys, json, os, time, shutil
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs' / 'hemodynamics.json').exists())
assert Path(sys.executable).resolve() == (ROOT / '.venv' / 'Scripts' / 'python.exe').resolve(), 'Select the project .venv kernel'
sys.path.insert(0, str(ROOT / 'notebooks'))
import importlib, psutil
import numpy as np, pandas as pd
from IPython.display import Image, Markdown, display
from hemodynamics import snapshot, eda_metrics as E, eda_models as M
for _module in (snapshot, M, E):   # pick up edited code in a long-lived kernel
    importlib.reload(_module)

me = psutil.Process()
for p in psutil.process_iter():
    try: p.cpu_percent(None)
    except psutil.Error: pass
time.sleep(0.8)
busy = []
for p in psutil.process_iter(['pid', 'name']):
    try:
        if p.pid != me.pid and p.info['name'] and 'python' in p.info['name'].lower() and p.cpu_percent(None) > 50:
            busy.append(p.pid)
    except psutil.Error:
        pass
print('Root:', ROOT)
print(f'Free disk: {shutil.disk_usage(ROOT).free / 1e9:.0f} GB | free RAM: {psutil.virtual_memory().available / 1e9:.0f} GB | CPUs: {os.cpu_count()}')
print('Other busy python processes:', busy or 'none')
print('Dataset tag:', snapshot.assert_dataset_v1(ROOT)['version'], '(the build refuses to run if the dataset changed)')
if busy:
    print('WARNING: another job is using the CPU; the throughput benchmark in section 4 would be misleading. Wait for it first.')

In [ ]:
res = E.build_all(ROOT, progress=True)                       # REGENERATE - START HERE (True = recomputed, False = cached)
print(res['stages_run'])
D, F = ROOT / 'results' / 'eda', ROOT / 'figures' / 'eda'
def table(name, rows=None, round_=4):
    t = pd.read_csv(D / f'{name}.csv')
    display(t.round(round_) if rows is None else t.head(rows).round(round_))
def fig(name):
    display(Image(filename=str(F / f'{name}.png')))
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 150)

## 1. Cohort, selection criteria and splits

In [ ]:
table('table_selection_criteria'); table('table_cohort_funnel'); fig('fig_cohort_funnel')
table('table_split_pooled_summary'); table('table_split_cases_patients'); table('table_split_windows'); fig('fig_split_windows')
table('table_mover_exclusion_reasons')

## 2. Signal coverage and quality
MOVER capture is intermittent; VitalDB is continuous. ECG->PPG delay and PPG device units differ between the datasets (a real dataset shift). QC gates are the 12-case detector check.

In [ ]:
table('table_signal_coverage'); table('table_gap_profile'); fig('fig_gap_profile')
table('table_beat_quality'); fig('fig_signal_quality'); table('table_qc_gates'); table('table_qc_per_case')

## 3. Labels
Event = MAP < 65 mmHg for >= 60 s starting within 300 s; windows already hypotensive are excluded; 65-75 mmHg is the gray zone (tagged); lead time is at most 240 s by the rule.

In [ ]:
table('table_labels_prevalence'); table('table_time_to_event'); table('table_label_distributions'); table('table_label_quality'); table('table_windows_per_patient'); fig('fig_labels')

## 4. Predictive baselines (AUROC)
Ladder rungs, all scored on the **identical BSHI-full windows**: G0 current MAP + 60 s trend (baseline A), G1 + ART morphology and HR, G2 + ECG->ART timing and HRV, G3 + PPG features and delays (PPG amplitude only as within-window relative features), G4 + SQI means and mask counts. Two learners per rung: standardized logistic regression (`lr`) and gradient boosting (`hgb`). Settings: pooled train -> pooled test, VitalDB train -> VitalDB test, **E3x** VitalDB train -> all MOVER, **reverse** MOVER train -> VitalDB test, MOVER test only (underpowered).

In [ ]:
fig('fig_auroc_ladder')
table('table_ladder_auroc_hgb_report_view')
t = pd.read_csv(D / 'table_baseline_ladder_all.csv')
full = t[t.set == 'full']
for setting in ['pooled_test', 'E3x vitaldb_train->all_mover', 'reverse mover_train->vitaldb_test', 'vitaldb_train->vitaldb_test', 'mover_test_only']:
    s = full[full.setting == setting].assign(CI=lambda d: d.auroc_lo.round(3).astype(str) + ' - ' + d.auroc_hi.round(3).astype(str))
    print(setting)
    display(s.pivot(index='rung', columns='learner', values=['auroc', 'CI', 'auprc', 'brier']).round(3))
print('Baseline A as saved (all eligible windows, not only BSHI-full):'); table('table_baseline_A_saved')

In [ ]:
print('AUROC by lead time (pooled test and E3x):'); table('table_auroc_by_lead_time')
print('Gray zone (65-75 mmHg) versus the rest:'); table('table_gray_zone')
print('Dead-ECG cases versus the rest (MAP-only model):'); table('table_dead_ecg_vs_rest')
fig('fig_roc_calibration_leadtime')

## 5. Dataset shift
A classifier that predicts the dataset from window features (patient-disjoint train/test): AUROC 0.5 means indistinguishable. Raw PPG amplitude and AC/DC are in device units (about 20x apart) and are excluded from every model and from the BSHI context.

In [ ]:
table('table_domain_shift'); table('table_smd', 15); fig('fig_domain_shift')

## 6. Readiness scorecard
Every check with its value, threshold and PASS/FAIL. MOVER-only test splits are underpowered (too few events and patients with an event): use all MOVER patients (E3x) as the external test and report MOVER-only numbers with their wide CIs.

In [ ]:
r = pd.read_csv(D / 'table_readiness.csv')
display(r[['check', 'value', 'threshold', 'status', 'note']].fillna('').style.map(lambda v: 'background-color: #c8f0c8' if v == 'PASS' else ('background-color: #f6c6c6' if v == 'FAIL' else ''), subset=['status']))
sc = json.loads((ROOT / 'results' / 'dataset_readiness.json').read_text())
print(f"{sc['n_pass']} PASS, {sc['n_fail']} FAIL")
print(json.dumps(sc['reproducibility'], indent=1))

## 7. Written summary (`docs/eda_summary.md`)

In [ ]:
display(Markdown((ROOT / 'docs' / 'eda_summary.md').read_text(encoding='utf-8')))